# 유사한 단어 찾기 게임

1. 사전 학습된 모델 또는 적절한 데이터셋을 찾는다.
2. 워드 임베딩 모델을 학습시킨다.
3. 단어 유사도가 0.8 이상인 A, B를 랜덤 추출한다.
4. A, B와 대응되는 C를 추출한다.
5. D를 입력 받는다.

=>
A:B = C:D 관계에 대응하는 D를 찾는 게임을 만든다.
ex) A: 산, B: 바다, C: 나무, D: 물

**<출력 예시>**

관계 [ 수긍 : 추락 = 대사관 : ? ]<br>
모델이 예측한 가장 적합한 단어: 잠입<br>
당신의 답변과 모델 예측의 유사도: 0.34<br>
아쉽네요. 더 생각해보세요.

In [1]:
import pandas as pd

splits = {'train': 'dp/train-00000-of-00001.parquet', 'validation': 'dp/validation-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/klue/klue/" + splits["train"])

In [2]:
df = df['sentence']

In [3]:
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

In [4]:
from konlpy.tag import Okt
from tqdm import tqdm

okt = Okt()
ko_stopwords = ["은", "는", "이", "가", "을", "를", "과", "와", "들", "도", "부터", "까지", "에", "나", "너", "그", "걔", "얘", "다"]

preprocessed_data = []

for sentence in tqdm(df):
    sentence = re.sub(r"[a-zA-Z]", " ", sentence)
    sentence = re.sub(r"[^가-힣\s]", " ", sentence)
    tokens = okt.morphs(sentence, stem=True)
    tokens = [token for token in tokens if token not in ko_stopwords and not token.endswith("다")]
    preprocessed_data.append(tokens)

100%|██████████| 10000/10000 [00:17<00:00, 587.64it/s]


In [5]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_data, # corpus
    vector_size=100,                  # 임베딩 벡터 차원
    sg=0,                             # 학습 알고리즘 (0:CBOW, 1:Skip-gram)
    window=5,                         # 주변 단어 수 (앞뒤로 n개 사용) -> 이게 왜 5로 설정했는지 다시 확인
    min_count=5                       # 최소 빈도
)

model.wv.vectors.shape

(3478, 100)

In [6]:
import pandas as pd

pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
의,-0.299447,0.513025,0.026726,-0.158662,0.260958,-0.594420,0.046783,1.154915,-0.229356,-0.060863,...,0.221516,0.340107,0.491690,0.051476,0.999718,0.492668,0.168317,-0.291879,-0.102828,0.337510
에서,-0.304262,0.599785,0.029743,-0.052679,0.098039,-0.645557,0.152025,1.062837,-0.182594,-0.201564,...,0.247558,0.343774,0.367015,0.129996,1.013613,0.328087,0.101692,-0.164916,-0.194151,0.239370
으로,-0.307679,0.448843,0.018361,-0.225850,0.423261,-0.454331,-0.027046,1.150178,-0.259966,0.034130,...,0.196077,0.322318,0.541538,-0.047375,0.942493,0.572155,0.257512,-0.321238,0.001264,0.365294
한,-0.324716,0.535997,0.007944,-0.181091,0.294529,-0.596185,0.042734,1.180496,-0.195911,-0.088998,...,0.202510,0.354322,0.507313,0.067538,1.042601,0.467870,0.207825,-0.291018,-0.133024,0.320285
로,-0.322095,0.539031,0.005664,-0.139763,0.229044,-0.608511,0.069850,1.109837,-0.231517,-0.100275,...,0.233049,0.327322,0.446715,0.099652,1.021455,0.405616,0.174921,-0.229650,-0.141447,0.326642
일,-0.245376,0.891301,-0.033448,0.114075,-0.459867,-0.696710,0.460635,0.880935,-0.023151,-0.555945,...,0.287902,0.499334,0.110796,0.367298,1.130156,-0.041871,-0.177832,0.141024,-0.419929,-0.001566
것,-0.283140,0.451514,0.044334,-0.184786,0.386574,-0.524726,0.000170,1.149150,-0.286330,0.025687,...,0.216918,0.288028,0.485890,-0.039726,0.920857,0.559673,0.278152,-0.332220,-0.037216,0.414475
숙소,-0.293530,0.490773,0.066840,-0.123623,0.260025,-0.632609,0.076680,1.147235,-0.294414,-0.062696,...,0.287373,0.314993,0.460330,0.031922,0.977540,0.498849,0.192066,-0.319797,-0.061363,0.365351
등,-0.330062,0.531474,0.032598,-0.162898,0.322097,-0.598832,0.018239,1.179748,-0.257255,-0.043634,...,0.228510,0.348113,0.513664,0.045020,1.038901,0.529745,0.218046,-0.295348,-0.086252,0.344890
씨,-0.309120,0.574193,-0.031921,-0.092594,0.087931,-0.629871,0.128062,1.099141,-0.147012,-0.222957,...,0.165700,0.426309,0.439838,0.207417,1.074395,0.286146,0.095904,-0.137476,-0.205065,0.236660


In [7]:
# 학습된 단어 임베딩 저장
model.wv.save_word2vec_format('all_kor_w2v')

In [8]:
# 임베딩 모델 로드
from gensim.models import KeyedVectors

load_model = KeyedVectors.load_word2vec_format('all_kor_w2v')

In [9]:
# model : Word2Vec
model.wv.most_similar('남자')

[('부', 0.999515175819397),
 ('특히', 0.9994581341743469),
 ('작업', 0.9994308948516846),
 ('운영', 0.9994089603424072),
 ('상', 0.9994085431098938),
 ('소속', 0.9994065761566162),
 ('사업', 0.9993782639503479),
 ('랑', 0.9993690848350525),
 ('병원', 0.9993609189987183),
 ('단', 0.9993494749069214)]

In [ ]:
import random

kv = load_model  # 이미 불러온 임베딩 사용

def play():
    vocab = list(kv.key_to_index.keys())

    # A, B 두 단어 랜덤 선택
    A, B = random.sample(vocab, 2)

    # A와 가장 유사한 단어 C 선택 (top1)
    try:
        C = kv.most_similar(A, topn=1)[0][0]
    except KeyError:
        print("해당 단어로는 유사도 계산 불가. 다시 실행하세요.")
        return

    # 모델 예측: A:B = C:?
    try:
        pred = kv.most_similar(positive=[B, C], negative=[A], topn=1)[0][0]
    except KeyError:
        print("관계 계산 불가. 다시 실행하세요.")
        return

    # 출력
    print(f"관계 [ {A} : {B} = {C} : ? ]")
    print(f"모델이 예측한 가장 적합한 단어: {pred}")

    # 사용자 입력
    user = input("D를 입력하세요: ").strip()
    if user in kv and pred in kv:
        sim = kv.similarity(user, pred)
        print(f"당신의 답변과 모델 예측의 유사도: {sim:.2f}")
    else:
        print("사전에 없는 단어라 유사도 계산 불가")


In [12]:
play()

관계 [ 다음 : 등급 = 방문 : ? ]
모델이 예측한 가장 적합한 단어: 이나
사전에 없는 단어라 유사도 계산 불가
